In [1]:
import re
import json
import time
import unicodedata
from typing import Optional, List, Dict
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from bs4 import BeautifulSoup

# --- Constants ---
APPROVED_URL        = "https://pdb.apec.org/Lists/Projects/Approved%20Projects.aspx"
DETAIL_URL_TEMPLATE = "https://pdb.apec.org/Lists/Projects/DispForm.aspx?ID={id}"
USER_AGENT          = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) APEC-Projects-Notebook/3.0"
MAX_WORKERS         = 50   # parallel detail-page fetchers
MAX_RETRIES         = 3    # attempts per detail page before giving up
RETRY_BACKOFF       = 2.0  # seconds to wait between retries (doubles each attempt)

# Shared session: connection pooling + persistent headers across all threads
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})


def soupify(html: str) -> BeautifulSoup:
    """Parse raw HTML into a BeautifulSoup tree."""
    return BeautifulSoup(html, "lxml")


def norm_text(s: Optional[str]) -> str:
    """NFKD-normalize unicode, replace typographic chars, collapse whitespace."""
    if s is None:
        return ""
    s = unicodedata.normalize("NFKD", s)
    s = s.replace("\u2019", "'").replace("\u2013", "-").replace("\u2014", "-")
    return re.sub(r"\s+", " ", s).strip()


def flatten_lookup_array(arr) -> str:
    """Flatten a SharePoint lookup array [{'lookupValue': '...'}] into 'a; b; c'."""
    if not isinstance(arr, list):
        return ""
    return "; ".join(
        str(v)
        for obj in arr
        if isinstance(obj, dict)
        for v in [obj.get("lookupValue") or obj.get("LookupValue", "")]
        if v
    )

In [2]:
def extract_wpq1_data(html: str):
    """
    Extract the embedded WPQ1ListData JSON from the SharePoint list page.
    The regex stops at 'var WPQ1SchemaData' to avoid over-capturing.
    Returns (rows, next_href) — next_href is None on the last page.
    """
    pattern = (
        r"var\s+WPQ1ListData\s*=\s*"
        r"(\{.*?\})"
        r"\s*;\s*var\s+WPQ1SchemaData"
    )
    m = re.search(pattern, html, re.DOTALL)
    if not m:
        print("WPQ1ListData not found on page.")
        return [], None

    try:
        data = json.loads(m.group(1))
    except json.JSONDecodeError as e:
        print("JSON decode error:", e)
        print(m.group(1)[:500])
        return [], None

    return data.get("Row", []), data.get("NextHref")


def fetch_all_approved_rows() -> List[Dict[str, str]]:
    """
    Paginate through the full Approved Projects list view using SharePoint's
    NextHref cursor. Returns a list of {id, link} dicts — one per project.
    Pagination is sequential; only detail fetching is parallelized later.
    """
    url = APPROVED_URL
    all_records: List[Dict[str, str]] = []
    seen_urls = set()  # guard against infinite loops if NextHref ever repeats
    page = 1

    while url and url not in seen_urls:
        print(f"[VIEW] Fetching page {page}: {url}")
        seen_urls.add(url)

        resp = session.get(url, timeout=40)
        resp.raise_for_status()

        rows, next_href = extract_wpq1_data(resp.text)
        print(f"  rows in this page: {len(rows)}")

        for r in rows:
            proj_id  = r.get("ID")
            file_ref = r.get("FileRef", "")
            # Prefer canonical DispForm URL; fall back to FileRef if ID is missing
            link = (
                DETAIL_URL_TEMPLATE.format(id=proj_id)
                if proj_id
                else urljoin("https://pdb.apec.org/", file_ref) if file_ref else ""
            )
            all_records.append({"id": proj_id, "link": link})

        if not next_href:
            print("  No NextHref -> last page reached.")
            break

        url = urljoin(APPROVED_URL, next_href)
        page += 1
        time.sleep(0.4)  # polite delay between list pages

    print(f"[VIEW] Total records found: {len(all_records)}")
    return all_records

In [3]:
# Label -> canonical column mapping.
# IMPORTANT: "Project Cost (US$)" belongs only to project_cost_usd.
# Self-funded projects use that field; TILF/Operational projects use
# apec_funding + co_funding_amount + total_project_value instead.
LABEL_ALIASES = {
    "project_title":                        ["Project Title", "Title"],
    "project_number":                       ["Project No.", "Project Number"],
    "project_year":                         ["Project Year"],
    "project_status":                       ["Project Status"],
    "fund_account":                         ["Fund Account"],
    "sub_fund":                             ["Sub-fund"],
    "project_session":                      ["Project Session"],
    "apec_funding":                         ["APEC Funding"],
    "co_funding_amount":                    ["Co-funding Amount", "Co-Funding Amount"],
    "total_project_value":                  ["Total Project Value"],
    "project_cost_usd":                     ["Project Cost (US$)"],
    "sponsoring_forum":                     ["Sponsoring Forum"],
    "topics":                               ["Topics"],
    "committee":                            ["Committee"],
    "other_fora_involved":                  ["Other Fora Involved"],
    "other_non_apec_stakeholders_involved": ["Other Non-APEC Stakeholders Involved"],
    "proposing_economy":                    ["Proposing Economy(ies)", "Proposing Economy"],
    "co_sponsoring_economies":              ["Co-Sponsoring Economies", "Co-sponsoring Economies"],
    "expected_start_date":                  ["Expected Start Date"],
    "expected_completion_date":             ["Expected Completion Date"],
    "organization_1":                       ["Organization 1"],
    "organization_2":                       ["Organization 2"],
}

# Final output column order
OUTPUT_COLUMNS = [
    "project_title", "project_number", "project_year", "sponsoring_forum",
    "proposing_economy", "project_status", "fund_account", "sub_fund",
    "project_session", "apec_funding", "co_funding_amount", "total_project_value",
    "project_cost_usd", "topics", "committee", "other_fora_involved",
    "co_sponsoring_economies", "organization_1", "organization_2", "link",
]

# Reverse lookup built once at module level: normalized label -> column name
ALIAS_INDEX = {
    re.sub(r"(\s*\*|:)\s*$", "", norm_text(alias)).lower(): col
    for col, aliases in LABEL_ALIASES.items()
    for alias in aliases
}


def _clean_label(txt: str) -> str:
    """Strip trailing asterisks and colons from a label string."""
    return re.sub(r"(\s*\*|:)\s*$", "", norm_text(txt))


def _text(el) -> str:
    """Return normalized inner text of a BS4 element, or '' if None."""
    return norm_text(el.get_text(" ", strip=True)) if el else ""


def extract_label_value_pairs(soup: BeautifulSoup) -> Dict[str, str]:
    """
    Extract all label/value pairs from a detail page.
    Tries three layout variants in priority order:
      1. Desktop  — div.lfi-fields-wrapper > .input-label + .input-col
      2. Fallback — p.lfi-field-label, then walks DOM to find .input-col
      3. Mobile   — .pdb-responsive-header paired with .pdb-responsive-content
    """
    pairs: Dict[str, str] = {}

    # 1) Desktop layout
    for wrap in soup.select("div.lfi-fields-wrapper"):
        lbl_div = wrap.select_one(".input-label")
        val_div = wrap.select_one(".input-col")
        if not lbl_div or not val_div:
            continue
        label = _clean_label(_text(lbl_div))
        if label:
            pairs[label] = _text(val_div)

    # 2) Fallback: label paragraph + sibling value div
    for lbl in soup.select("p.lfi-field-label"):
        label = _clean_label(_text(lbl))
        if not label or label in pairs:  # skip if already captured above
            continue
        val_div = None
        cont = lbl.find_parent(class_=re.compile(r"input-label"))
        if cont:
            # Try the direct sibling first (fastest path)
            sib = cont.find_next_sibling(class_=re.compile(r"input-col"))
            if sib:
                val_div = sib
        if not val_div:
            # Walk up to 25 DOM nodes forward to find the value container
            steps, nxt = 0, lbl
            while nxt and steps < 25 and not val_div:
                nxt = nxt.next_element
                steps += 1
                if getattr(nxt, "name", "") and re.search(
                    r"\binput-col\b", " ".join(nxt.get("class", []))
                ):
                    val_div = nxt
                    break
        value = _text(val_div) if val_div else ""
        if label and value:
            pairs[label] = value

    # 3) Mobile / responsive layout — headers and content are parallel lists
    headers  = soup.select(".pdb-responsive-header")
    contents = soup.select(".pdb-responsive-content")
    if headers and contents and len(headers) == len(contents):
        for h, c in zip(headers, contents):
            label = _clean_label(_text(h))
            if label and label not in pairs:
                pairs[label] = _text(c)

    return pairs


def parse_detail(url: str) -> dict:
    """
    Fetch a project detail page with up to MAX_RETRIES attempts.
    Uses exponential backoff on failure. Returns a dict of canonical
    columns; missing fields default to empty string.
    """
    if not url:
        return {}

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = session.get(url, headers={"Referer": APPROVED_URL}, timeout=40)
            r.raise_for_status()
            break  # success — exit retry loop
        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES:
                wait = RETRY_BACKOFF * attempt  # 2s, 4s, 6s
                print(f"  [RETRY {attempt}/{MAX_RETRIES}] {url} — {e} (waiting {wait}s)")
                time.sleep(wait)
    else:
        # All attempts exhausted — return empty row so the project isn't lost
        print(f"  [FAIL] giving up on {url}: {last_err}")
        return {}

    # Map raw label/value pairs to canonical column names
    raw = extract_label_value_pairs(soupify(r.text))
    out: Dict[str, str] = {}
    for k, v in raw.items():
        key_norm = _clean_label(k).lower()
        if key_norm in ALIAS_INDEX:
            out[ALIAS_INDEX[key_norm]] = v

    # Fill any missing columns with empty string
    for col in LABEL_ALIASES:
        out.setdefault(col, "")

    return out

In [4]:
def fetch_details_parallel(base_records: List[Dict]) -> List[Dict]:
    """
    Fetch all detail pages concurrently using a thread pool.
    Results are collected in original order via a future->index map.
    Progress is reported every 10 completed pages.
    """
    total   = len(base_records)
    results = [None] * total  # pre-allocated list preserves insertion order
    done    = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit all tasks; map each future back to its original index
        future_to_idx = {
            executor.submit(parse_detail, row["link"]): (i, row["link"])
            for i, row in enumerate(base_records)
        }

        for future in as_completed(future_to_idx):
            idx, link = future_to_idx[future]
            det = future.result()     # re-raises any uncaught exception from the worker
            det["link"] = link        # attach the source URL to the row
            results[idx] = det        # place result at original position
            done += 1
            if done % 10 == 0 or done == total:
                print(f"[DETAIL] {done}/{total} pages fetched...")

    return results


def main():
    # Phase 1: paginate the list view to collect all project links (sequential)
    base_records = fetch_all_approved_rows()
    if not base_records:
        print("No base records found. Exiting.")
        return

    # Phase 2: fetch all detail pages in parallel
    records = fetch_details_parallel(base_records)

    # Build DataFrame and enforce column schema
    df = pd.DataFrame.from_records(records)
    for col in OUTPUT_COLUMNS:
        if col not in df.columns:
            df[col] = ""

    df = (
        df[OUTPUT_COLUMNS]
        .replace("", pd.NA)           # empty strings -> NaN for cleaner Excel output
        .drop_duplicates(keep="first")
        .reset_index(drop=True)
    )

    print("Final shape:", df.shape)
    print(df.head(5))

    out_base = "apec_approved_projects"
    df.to_csv(f"{out_base}.csv", index=False, encoding="utf-8-sig")
    df.to_excel(f"{out_base}.xlsx", index=False)
    print(f"Saved: {out_base}.csv, {out_base}.xlsx")


if __name__ == "__main__":
    main()

[VIEW] Fetching page 1: https://pdb.apec.org/Lists/Projects/Approved%20Projects.aspx
  rows in this page: 20
[VIEW] Fetching page 2: https://pdb.apec.org/Lists/Projects/Approved%20Projects.aspx?Paged=TRUE&p_Modified=20260219%2006%3a27%3a03&p_ID=16066&PageFirstRow=21&View=ed480244-96d1-46d9-8f53-ce01eb91af69
  rows in this page: 20
[VIEW] Fetching page 3: https://pdb.apec.org/Lists/Projects/Approved%20Projects.aspx?Paged=TRUE&p_Modified=20260210%2007%3a49%3a54&p_ID=14924&PageFirstRow=41&View=ed480244-96d1-46d9-8f53-ce01eb91af69
  rows in this page: 20
[VIEW] Fetching page 4: https://pdb.apec.org/Lists/Projects/Approved%20Projects.aspx?Paged=TRUE&p_Modified=20260121%2014%3a39%3a30&p_ID=15917&PageFirstRow=61&View=ed480244-96d1-46d9-8f53-ce01eb91af69
  rows in this page: 20
[VIEW] Fetching page 5: https://pdb.apec.org/Lists/Projects/Approved%20Projects.aspx?Paged=TRUE&p_Modified=20260109%2016%3a54%3a36&p_ID=15987&PageFirstRow=81&View=ed480244-96d1-46d9-8f53-ce01eb91af69
  rows in this page